# Regression Failure Investigation

This notebook checks which model specs succeeded or failed, then inspects the firm + state fixed-effects designs for sparsity, separation, and other convergence issues.

In [5]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import statsmodels.api as sm

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.regression_models import (
    prepare_firm_fixed_effects,
    region,
    industry,
    firm,
    education,
    year,
    experience,
    benefits4,
)
from src.package_files.logit_model import run_logit_model

In [6]:
summary_path = repo_root / 'results/tables_2026/model_panels_ai_role_long.csv'
summary = pd.read_csv(summary_path)
summary['is_error'] = summary['model_status'].ne('ok')
display(summary.groupby(['panel', 'model_label'])['model_status'].first().reset_index())
display(summary.pivot_table(index='benefit_label', columns=['panel', 'model_label'], values='model_status', aggfunc='first'))

,panel,model_label,model_status
0,P1,M1 Baseline (Year+Industry),ok
1,P1,M2 + Individual Controls,ok
2,P1,M3 + Salary,ok
3,P1,M4 + Firm+State FE,error
4,P2,M1 Baseline (Year+Firm+State),error
5,P2,M2 + Individual Controls,error


panel                                         P1                           \
model_label          M1 Baseline (Year+Industry) M2 + Individual Controls   
benefit_label                                                               
Health and Wellbeing                          ok                       ok   
Paid Leave                                    ok                       ok   
Parental Leave                                ok                       ok   
Remote Work                                   ok                       ok   
Tuition Assistance                            ok                       ok   
Inclusive Workplace                             ok                       ok   

panel                                                \
model_label          M3 + Salary M4 + Firm+State FE   
benefit_label                                         
Health and Wellbeing          ok              error   
Paid Leave                    ok              error   
Parental Leave                ok              error   
Remote Work                   ok              error   
Tuition Assistance            ok              error   
Inclusive Workplace             ok              error   

panel                                           P2                           
model_label          M1 Baseline (Year+Firm+State) M2 + Individual Controls  
benefit_label                                                                
Health and Wellbeing                         error                    error  
Paid Leave                                      ok                    error  
Parental Leave                               error                    error  
Remote Work                                  error                    error  
Tuition Assistance                           error                    error  
Inclusive Workplace                            error                    error

In [7]:
base = repo_root / 'data/processed'
data = pd.read_parquet(base / 'labeled_v1.parquet')
remote_df = pd.read_parquet(base / 'labeled_v2.parquet')
data = data.merge(remote_df[['ID', 'REMOTE_KW']], on='ID', how='left')

bins = [-2, -1, 0, 2, 5, 10, 20, 100]
labels = ['Missing', '0 years', '1-2 years', '3-5 years', '6-10 years', '11-20 years', '21+ years']
data['EXPERIENCE_BUCKET'] = pd.cut(data['MIN_YEARS_EXPERIENCE'], bins=bins, labels=labels, right=True).astype(str).replace('nan', 'None Listed')
data['LOG_SALARY'] = np.log(data['SALARY'])
display(data[['YEAR', 'SALARY', 'LOG_SALARY', 'EXPERIENCE_BUCKET']].head())
print(data.shape)

,YEAR,SALARY,LOG_SALARY,EXPERIENCE_BUCKET
0,2021,NaN,NaN,3-5 years
1,2021,NaN,NaN,3-5 years
2,2022,100000.0,11.512925,3-5 years
3,2022,NaN,NaN,None Listed
4,2022,NaN,NaN,1-2 years


(99860, 38)


In [9]:
failed_specs = [
    ('P1 M4: +Firm+State FE', [year, firm, region, education, experience], ['LOG_SALARY']),
    ('P2 M1: Firm+State Baseline', [year, firm, region], []),
    ('P2 M2: +Indiv Controls', [year, firm, region, education, experience], []),
]

def spec_frame(df, dep, cat_controls, cont_controls):
    x = df[['AI ROLE']].copy()
    for cc in cont_controls:
        x = pd.concat([x, df[[cc]]], axis=1)
    for c in cat_controls:
        d = pd.get_dummies(df[c].astype('category'), drop_first=True)
        x = pd.concat([x, d], axis=1)
    y = df[dep]
    out = pd.concat([x, y], axis=1).dropna()
    return out

for label, cat_controls, cont_controls in failed_specs:
    print('\n' + '=' * 80)
    print(label)
    for dep in benefits4:
        # Inspect stats on the raw data (before dummy expansion)
        cols_needed = ['AI ROLE', dep] + cat_controls + cont_controls
        raw = data[cols_needed].dropna()
        print(f'\nBenefit: {dep}')
        print('Rows:', len(raw), ' | Outcome rate:', round(raw[dep].mean(), 4))
        if firm in cat_controls:
            print('Firm categories (raw):', raw[firm].nunique())
        if region in cat_controls:
            print('State categories:', raw[region].nunique())
        if education in cat_controls:
            print('Education categories:', raw[education].nunique())
            print(raw[education].value_counts())
        if experience in cat_controls:
            print(raw[experience].value_counts())



P1 M4: +Firm+State FE

Benefit: EDU_ASSISTANCE
Rows: 35245  | Outcome rate: 0.1107
Firm categories (raw): 17682
State categories: 51
Education categories: 6
MIN_EDULEVELS_NAME
No Education Listed             17368
High school or GED               9179
Bachelor's degree                6002
Associate degree                 1730
Master's degree                   694
Ph.D. or professional degree      272
Name: count, dtype: int64
EXPERIENCE_BUCKET
None Listed    19412
1-2 years       7465
3-5 years       5049
6-10 years      1581
0 years         1493
11-20 years      245
Name: count, dtype: int64

Benefit: PAID LEAVE
Rows: 35245  | Outcome rate: 0.4135
Firm categories (raw): 17682
State categories: 51
Education categories: 6
MIN_EDULEVELS_NAME
No Education Listed             17368
High school or GED               9179
Bachelor's degree                6002
Associate degree                 1730
Master's degree                   694
Ph.D. or professional degree      272
Name: count, dtype: i

In [10]:
# Optional diagnostic: try a higher firm threshold to see whether collapsing sparse firms helps.
data_fe_100 = prepare_firm_fixed_effects(data, min_firm_obs=100)
print(data_fe_100[firm].value_counts().head())

# Uncomment one line below to test a single failed spec after collapsing firms.
# model = run_logit_model(data_fe_100, dependent='EDU_ASSISTANCE', predictor='AI ROLE', cat_controls=[year, firm, region, education, experience], cont_controls=['LOG_SALARY'], ref_category={education: 'No Education Listed', experience: 'None Listed'}, get_vif=False)
# print(model.summary())

Firm FE prep: 35,081 firms -> 49 categories after grouping firms with <100 obs
COMPANY
Other Firm (<30 obs)    78166
0                       12855
20                        569
38121853                  453
36704772                  405
Name: count, dtype: int64


## What to look for

If the failing specs have very few positive outcomes inside some firm/state/education/experience cells, the issue is structural separation rather than code. Try the optional reduced-fit cell after increasing the firm grouping threshold, or simplify the full spec by dropping one FE block at a time.